# Transformers from Scratch

**Module 07: Transformers**  
**Objective**: Build the Transformer architecture from first principles

**"Attention Is All You Need" (Vaswani et al., 2017)**

Transformers revolutionized NLP:
- No recurrence → parallel training
- Self-attention → capture long-range dependencies
- Positional encoding → maintain sequence order

## What You'll Learn
1. Positional encoding
2. Multi-head self-attention
3. Feed-forward networks
4. Layer normalization
5. Encoder-decoder architecture
6. Complete Transformer implementation

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import math

np.random.seed(42)

## 1. Positional Encoding

**Problem**: Self-attention has no notion of position

**Solution**: Add positional information to embeddings

$$PE_{(pos, 2i)} = \sin(pos / 10000^{2i/d_{model}})$$
$$PE_{(pos, 2i+1)} = \cos(pos / 10000^{2i/d_{model}})$$

**Properties**:
- Unique for each position
- Same distance between consecutive positions
- Generalizes to longer sequences

In [ ]:
def positional_encoding(seq_len, d_model):
    """
    Generate positional encoding.
    
    Args:
        seq_len: sequence length
        d_model: model dimension
    
    Returns:
        PE: (seq_len, d_model) positional encodings
    """
    PE = np.zeros((seq_len, d_model))
    
    for pos in range(seq_len):
        for i in range(0, d_model, 2):
            # Compute frequency
            div_term = 10000 ** (2 * i / d_model)
            
            # Sin for even indices
            PE[pos, i] = np.sin(pos / div_term)
            
            # Cos for odd indices
            if i + 1 < d_model:
                PE[pos, i + 1] = np.cos(pos / div_term)
    
    return PE

# Generate and visualize
seq_len = 100
d_model = 128

PE = positional_encoding(seq_len, d_model)

print(f"Positional encoding shape: {PE.shape}\n")

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Heatmap
sns.heatmap(PE.T, cmap='coolwarm', center=0, ax=axes[0])
axes[0].set_xlabel('Position')
axes[0].set_ylabel('Dimension')
axes[0].set_title('Positional Encoding Heatmap')

# Line plots for different positions
axes[1].plot(PE[0, :], label='Position 0')
axes[1].plot(PE[10, :], label='Position 10')
axes[1].plot(PE[50, :], label='Position 50')
axes[1].set_xlabel('Dimension')
axes[1].set_ylabel('Value')
axes[1].set_title('Positional Encoding for Different Positions')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Show properties
print("Key properties:")
print(f"1. Unique encoding for each position")
print(f"2. Values in [-1, 1]")
print(f"3. PE[0][0:10] = {PE[0, :10]}")
print(f"4. PE[50][0:10] = {PE[50, :10]}")

## 2. Scaled Dot-Product Attention (Review)

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

**Steps**:
1. Compute attention scores: $QK^T / \sqrt{d_k}$
2. Apply softmax to get weights
3. Weighted sum of values

In [ ]:
def scaled_dot_product_attention(Q, K, V, mask=None):
    """
    Scaled dot-product attention.
    
    Args:
        Q: (batch, heads, seq_len_q, d_k)
        K: (batch, heads, seq_len_k, d_k)
        V: (batch, heads, seq_len_v, d_v)
        mask: optional mask
    
    Returns:
        output, attention_weights
    """
    d_k = Q.shape[-1]
    
    # Attention scores
    scores = np.matmul(Q, K.transpose(0, 1, 3, 2)) / np.sqrt(d_k)
    
    # Apply mask
    if mask is not None:
        scores = scores + mask * -1e9
    
    # Softmax
    exp_scores = np.exp(scores - np.max(scores, axis=-1, keepdims=True))
    attention_weights = exp_scores / np.sum(exp_scores, axis=-1, keepdims=True)
    
    # Apply to values
    output = np.matmul(attention_weights, V)
    
    return output, attention_weights

# Test
batch, heads, seq_len, d_k = 2, 8, 10, 64
Q = np.random.randn(batch, heads, seq_len, d_k)
K = np.random.randn(batch, heads, seq_len, d_k)
V = np.random.randn(batch, heads, seq_len, d_k)

output, attn = scaled_dot_product_attention(Q, K, V)
print(f"Attention output shape: {output.shape}")
print(f"Attention weights shape: {attn.shape}")

## 3. Multi-Head Attention

**Idea**: Multiple attention heads learn different representations

$$\text{MultiHead}(Q, K, V) = \text{Concat}(\text{head}_1, ..., \text{head}_h)W^O$$

where $\text{head}_i = \text{Attention}(QW_i^Q, KW_i^K, VW_i^V)$

**Benefits**:
- Attending to different positions
- Different representation subspaces
- More expressive

In [ ]:
class MultiHeadAttention:
    """Multi-head attention from scratch."""
    
    def __init__(self, d_model, num_heads):
        assert d_model % num_heads == 0
        
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        
        # Linear projections
        self.W_q = np.random.randn(d_model, d_model) * 0.01
        self.W_k = np.random.randn(d_model, d_model) * 0.01
        self.W_v = np.random.randn(d_model, d_model) * 0.01
        self.W_o = np.random.randn(d_model, d_model) * 0.01
    
    def split_heads(self, x, batch_size):
        """Split into multiple heads."""
        # (batch, seq_len, d_model) → (batch, seq_len, heads, d_k)
        x = x.reshape(batch_size, -1, self.num_heads, self.d_k)
        # → (batch, heads, seq_len, d_k)
        return x.transpose(0, 2, 1, 3)
    
    def forward(self, Q, K, V, mask=None):
        """
        Multi-head attention forward.
        
        Args:
            Q, K, V: (batch, seq_len, d_model)
            mask: optional mask
        
        Returns:
            output: (batch, seq_len, d_model)
            attention_weights: (batch, heads, seq_len, seq_len)
        """
        batch_size = Q.shape[0]
        
        # Linear projections
        Q = np.matmul(Q, self.W_q)
        K = np.matmul(K, self.W_k)
        V = np.matmul(V, self.W_v)
        
        # Split heads
        Q = self.split_heads(Q, batch_size)  # (batch, heads, seq_len, d_k)
        K = self.split_heads(K, batch_size)
        V = self.split_heads(V, batch_size)
        
        # Scaled dot-product attention
        output, attention_weights = scaled_dot_product_attention(Q, K, V, mask)
        
        # Concatenate heads
        output = output.transpose(0, 2, 1, 3)  # (batch, seq_len, heads, d_k)
        output = output.reshape(batch_size, -1, self.d_model)  # (batch, seq_len, d_model)
        
        # Final linear
        output = np.matmul(output, self.W_o)
        
        return output, attention_weights

# Test multi-head attention
print("\nTesting Multi-Head Attention\n")

d_model = 512
num_heads = 8
batch_size = 2
seq_len = 10

mha = MultiHeadAttention(d_model, num_heads)

X = np.random.randn(batch_size, seq_len, d_model)
output, attn_weights = mha.forward(X, X, X)

print(f"Input shape: {X.shape}")
print(f"Output shape: {output.shape}")
print(f"Attention weights shape: {attn_weights.shape}")
print(f"d_k per head: {mha.d_k}\n")

# Visualize attention patterns
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for head in range(num_heads):
    sns.heatmap(attn_weights[0, head], cmap='Blues', cbar=False, ax=axes[head])
    axes[head].set_title(f'Head {head + 1}')
    axes[head].set_xlabel('Key')
    axes[head].set_ylabel('Query')

plt.suptitle('Multi-Head Attention Patterns', fontsize=16)
plt.tight_layout()
plt.show()

## 4. Feed-Forward Network

**Position-wise FFN**: Applied independently to each position

$$\text{FFN}(x) = \text{ReLU}(xW_1 + b_1)W_2 + b_2$$

**Typical dimensions**: $d_{model} = 512$, $d_{ff} = 2048$

**Purpose**: Add non-linearity and capacity

In [ ]:
class FeedForward:
    """Position-wise feed-forward network."""
    
    def __init__(self, d_model, d_ff):
        self.d_model = d_model
        self.d_ff = d_ff
        
        self.W1 = np.random.randn(d_model, d_ff) * np.sqrt(2.0 / d_model)
        self.b1 = np.zeros(d_ff)
        self.W2 = np.random.randn(d_ff, d_model) * np.sqrt(2.0 / d_ff)
        self.b2 = np.zeros(d_model)
    
    def forward(self, x):
        """
        Forward pass.
        
        Args:
            x: (batch, seq_len, d_model)
        
        Returns:
            output: (batch, seq_len, d_model)
        """
        # First layer + ReLU
        hidden = np.maximum(0, np.matmul(x, self.W1) + self.b1)
        
        # Second layer
        output = np.matmul(hidden, self.W2) + self.b2
        
        return output

# Test FFN
ffn = FeedForward(d_model=512, d_ff=2048)

x = np.random.randn(2, 10, 512)
output = ffn.forward(x)

print(f"FFN input shape: {x.shape}")
print(f"FFN output shape: {output.shape}")
print(f"Parameters: W1={(512, 2048)}, W2={(2048, 512)}")
print(f"Total FFN params: {512*2048 + 2048 + 2048*512 + 512:,}")

## 5. Layer Normalization

**Batch Norm**: Normalize across batch dimension  
**Layer Norm**: Normalize across feature dimension (better for sequences)

$$\text{LayerNorm}(x) = \gamma \frac{x - \mu}{\sqrt{\sigma^2 + \epsilon}} + \beta$$

where $\mu, \sigma^2$ computed over last dimension

In [ ]:
class LayerNorm:
    """Layer normalization."""
    
    def __init__(self, d_model, eps=1e-6):
        self.d_model = d_model
        self.eps = eps
        
        self.gamma = np.ones(d_model)
        self.beta = np.zeros(d_model)
    
    def forward(self, x):
        """
        Args:
            x: (batch, seq_len, d_model)
        
        Returns:
            normalized: (batch, seq_len, d_model)
        """
        mean = np.mean(x, axis=-1, keepdims=True)
        std = np.std(x, axis=-1, keepdims=True)
        
        normalized = (x - mean) / (std + self.eps)
        
        # Scale and shift
        output = self.gamma * normalized + self.beta
        
        return output

# Test LayerNorm
ln = LayerNorm(d_model=512)

x = np.random.randn(2, 10, 512) * 10 + 50  # Large mean and variance
output = ln.forward(x)

print(f"\nLayer Normalization Test:")
print(f"Input mean: {np.mean(x):.2f}, std: {np.std(x):.2f}")
print(f"Output mean: {np.mean(output):.4f}, std: {np.std(output):.4f}")
print(f"✓ LayerNorm normalizes distribution!")

## 6. Encoder Layer

**Transformer Encoder Layer**:
1. Multi-head self-attention
2. Add & Norm (residual connection + layer norm)
3. Position-wise FFN
4. Add & Norm

$$\text{Encoder}(x) = \text{LayerNorm}(x + \text{FFN}(\text{LayerNorm}(x + \text{MultiHead}(x))))$$

In [ ]:
class TransformerEncoderLayer:
    """Single Transformer encoder layer."""
    
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        self.self_attn = MultiHeadAttention(d_model, num_heads)
        self.ffn = FeedForward(d_model, d_ff)
        self.norm1 = LayerNorm(d_model)
        self.norm2 = LayerNorm(d_model)
        self.dropout = dropout
    
    def forward(self, x, mask=None):
        """
        Encoder layer forward.
        
        Args:
            x: (batch, seq_len, d_model)
            mask: optional mask
        
        Returns:
            output: (batch, seq_len, d_model)
        """
        # Self-attention
        attn_output, attn_weights = self.self_attn.forward(x, x, x, mask)
        
        # Add & Norm
        x = self.norm1.forward(x + attn_output)
        
        # FFN
        ffn_output = self.ffn.forward(x)
        
        # Add & Norm
        output = self.norm2.forward(x + ffn_output)
        
        return output, attn_weights

# Test encoder layer
print("\nTesting Transformer Encoder Layer\n")

encoder_layer = TransformerEncoderLayer(d_model=512, num_heads=8, d_ff=2048)

x = np.random.randn(2, 10, 512)
output, attn = encoder_layer.forward(x)

print(f"Input shape: {x.shape}")
print(f"Output shape: {output.shape}")
print(f"Attention weights shape: {attn.shape}")

## 7. Complete Transformer Encoder

In [ ]:
class TransformerEncoder:
    """Stack of N encoder layers."""
    
    def __init__(self, num_layers, d_model, num_heads, d_ff, dropout=0.1):
        self.num_layers = num_layers
        self.layers = [TransformerEncoderLayer(d_model, num_heads, d_ff, dropout) 
                      for _ in range(num_layers)]
        self.norm = LayerNorm(d_model)
    
    def forward(self, x, mask=None):
        """
        Encode input sequence.
        
        Args:
            x: (batch, seq_len, d_model) embeddings + positional encoding
            mask: optional mask
        
        Returns:
            output: (batch, seq_len, d_model)
            all_attention_weights: list of attention weights from each layer
        """
        all_attention_weights = []
        
        for layer in self.layers:
            x, attn = layer.forward(x, mask)
            all_attention_weights.append(attn)
        
        output = self.norm.forward(x)
        
        return output, all_attention_weights

# Build complete encoder
print("\nBuilding Complete Transformer Encoder\n")

NUM_LAYERS = 6
D_MODEL = 512
NUM_HEADS = 8
D_FF = 2048

encoder = TransformerEncoder(NUM_LAYERS, D_MODEL, NUM_HEADS, D_FF)

# Test with embeddings + positional encoding
batch_size = 2
seq_len = 20
vocab_size = 10000

# Simulate token embeddings
token_embeddings = np.random.randn(batch_size, seq_len, D_MODEL) * 0.1

# Add positional encoding
PE = positional_encoding(seq_len, D_MODEL)
x = token_embeddings + PE  # Broadcast PE to batch

# Encode
output, all_attn = encoder.forward(x)

print(f"Config: {NUM_LAYERS} layers, d_model={D_MODEL}, heads={NUM_HEADS}, d_ff={D_FF}")
print(f"\nInput shape: {x.shape}")
print(f"Output shape: {output.shape}")
print(f"Number of attention weight tensors: {len(all_attn)}")

# Visualize attention from last layer
last_layer_attn = all_attn[-1][0]  # (heads, seq_len, seq_len)

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for head in range(8):
    sns.heatmap(last_layer_attn[head], cmap='Blues', cbar=False, ax=axes[head])
    axes[head].set_title(f'Layer {NUM_LAYERS}, Head {head+1}')
    axes[head].set_xlabel('Key Position')
    axes[head].set_ylabel('Query Position')

plt.suptitle('Transformer Encoder: Final Layer Attention Patterns', fontsize=16)
plt.tight_layout()
plt.show()

## Summary

You've built the Transformer encoder from scratch:
- ✅ Positional encoding (sine/cosine)
- ✅ Scaled dot-product attention
- ✅ Multi-head attention
- ✅ Position-wise feed-forward network
- ✅ Layer normalization
- ✅ Residual connections
- ✅ Complete Transformer encoder

**Key insights**:
1. **No recurrence**: Fully parallelizable (unlike RNNs)
2. **Self-attention**: Each token attends to all others
3. **Positional encoding**: Maintains sequence order
4. **Multi-head**: Different heads learn different patterns
5. **Residual + LayerNorm**: Stabilizes deep networks

**Transformer variants**:
- **Encoder-only**: BERT (bidirectional, classification)
- **Decoder-only**: GPT (causal/autoregressive, generation)
- **Encoder-decoder**: T5 (translation, summarization)

**Next Notebook**: Implementing Transformers with PyTorch and using pretrained models (BERT, GPT)

## Exercises

1. **Causal Masking**: Implement decoder with causal self-attention (mask future positions)
2. **Cross-Attention**: Add cross-attention layer for encoder-decoder architecture
3. **Relative Positional Encoding**: Implement relative position representations (T5-style)